In [1]:

import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
import QMIX
import buffer
import pong
from easydict import EasyDict
import jax
import jax.numpy as jnp
from physics import *
from util import *
config = EasyDict()
config.dt = 0.05
config.percent = 0.2
config.slop = 0.01
config.restitution = 1.0
config.state_dim = 13
config.action_dim = 2
config.batch_size = 256
config.episode_length = 1000
config.sync_period = 100
config.n_agents = 2
config.embed_dim = 64
config.layer_dim = 256
config.tau = 0.005
config.gamma = 0.99
config.lr = 1e-3
config.seed = 0

config.ppo_epochs = 10
config.clip_ratio = 0.05
config.entropy_coef = 0.0

In [2]:
pong_env = pong.Pong(config)
obs, state = pong_env.reset(jax.random.key(config.seed))

import tensorflow_probability.substrates.jax as tfp
tfd = tfp.distributions
tfb = tfp.bijectors


def sample_action(policy, states, key):
    logits = policy(states)
    sample = tfd.Categorical(logits = logits).sample(seed = key)
    log_probs = tfd.Categorical(logits = logits).log_prob(sample)
    return sample, log_probs

import MAPPO

train_state = MAPPO.init_train_state(config)

from flax import nnx

NetworkState = namedtuple('NetworkState', ['graphdef', 'state'])
TrainState = namedtuple('TrainState', ['pi1_state', 'pi2_state', 'value_state', 'key'])
Model = namedtuple('Model', ['network', 'optimizer'])

def get_model(state: NetworkState) -> Model:
    network, optimizer = nnx.merge(state.graphdef, state.state)
    return network, optimizer


def sample_action(policy, states, key):
    logits = policy(states)
    sample = tfd.Categorical(logits = logits).sample(seed = key)
    log_probs = tfd.Categorical(logits = logits).log_prob(sample)
    return sample, log_probs


2025-03-18 10:21:00.367667: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1742293260.379333 1866091 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1742293260.383056 1866091 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [3]:
def rollout(train_state):

    key = train_state.key
        
    pi1, pi1_optimizer = get_model(train_state.pi1_state)
    pi2, pi2_optimizer = get_model(train_state.pi2_state)
    value, value_optimizer = get_model(train_state.value_state)

    key, env_reset_key = jax.random.split(key, 2)
    def rollout_(carry, _):
        (env_state, obs, key) = carry
        key1, key2, key3, key4, env_reset_key, key = jax.random.split(key, 6)
        a1, log_pi1 = sample_action(pi1, obs['agent_0'], key1)
        a2, log_pi2 = sample_action(pi2, obs['agent_1'], key2)
        a3, log_pi3 = sample_action(pi2, obs['agent_2'], key3)
        a4, log_pi4 = sample_action(pi1, obs['agent_3'], key4)

        next_obs, next_state, reward, done, info = pong_env.step(None, env_state, {
            'agent_0': a1[None],
            'agent_1': a2[None],
            'agent_2': a3[None],
            'agent_3': a4[None]
        }, config)

        init_obs, init_state = pong_env.reset(env_reset_key)

        is_reset = done | (info['timestep'] > config.episode_length)
        next_obs = jax.tree.map(lambda x, y : jnp.where(is_reset, x, y), init_obs, next_obs)
        next_state_ = jax.tree.map(lambda x, y : jnp.where(is_reset, x, y), init_state, next_state)

        return (next_state_, next_obs, key), (env_state,obs, reward, done, log_pi1, log_pi2, log_pi3, log_pi4, a1, a2, a3, a4)
    
    obs, state = pong_env.reset(env_reset_key)

    (env_state, _, key), (env_state, obs, reward, done, log_pi1, log_pi2, log_pi3, log_pi4, a1, a2, a3, a4) = jax.lax.scan(rollout_, (state, obs, key), None, config.n_rollout)
    left_values = value(obs['agent_0'])
    right_values = value(obs['agent_3'])
    train_state = train_state._replace(key = key)
    return train_state, (env_state, obs, reward, done, log_pi1, log_pi2, log_pi3, log_pi4, a1, a2, a3, a4, left_values, right_values)

In [4]:
config.n_rollout = 1000

In [5]:

train_state, (env_state, obs, reward, done, log_pi1, log_pi2, log_pi3, log_pi4, a1, a2, a3, a4, left_values, right_values) = rollout(train_state)

In [6]:
env_state['ball'].texture.color.mean(axis=0)

Array([0.95100003, 0.        , 0.049     ], dtype=float32)

In [11]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from pong import Ball, Paddle, GoalLine, StaticBox
import numpy as np
def visualize_frame(objects, ax, frame_num=None):
    """현재 프레임의 모든 물체를 시각화합니다."""
    ax.clear()
    ax.set_xlim(-5, 5)
    ax.set_ylim(-3, 3)
    ax.set_aspect('equal')
    
    # 현재 프레임 번호와 점수 표시
    title_text = f"Frame: {frame_num}"
    ax.set_title(title_text)
    
    for obj in objects.values():
        
        if hasattr(obj, 'collider'):
            if isinstance(obj.collider, CircleCollider):
                circle = plt.Circle(
                    (obj.transform.position[0], obj.transform.position[1]),
                    obj.collider.radius,
                    color=np.array(obj.texture.color),
                    fill=True
                )
                ax.add_patch(circle)
            elif isinstance(obj.collider, BoxCollider):
                rectangle = patches.Rectangle(
                    (obj.transform.position[0] - obj.collider.width / 2, obj.transform.position[1] - obj.collider.height / 2),
                    obj.collider.width,
                    obj.collider.height,
                    color=np.array(obj.texture.color),
                    fill=True
                )
                ax.add_patch(rectangle)
        
    return ax.patches

def create_animation(history, interval=50, save_gif=False, filename='simulation.gif', frame_skip=1):
    """시뮬레이션 결과의 애니메이션을 생성합니다."""
    fig, ax = plt.subplots(figsize=(8, 8))
    
    # 프레임 스킵 적용
    total_frames = len(history['ball'].transform.position)
    frames_to_show = range(0, total_frames, frame_skip)
    
    def update(frame_idx):
        frame = frames_to_show[frame_idx]
        return visualize_frame(jax.tree.map(lambda x: x[frame], history), ax, frame_num=frame)
    
    ani = FuncAnimation(
        fig, 
        update, 
        frames=len(frames_to_show),
        interval=interval,
        blit=True
    )
    
    # GIF로 저장
    if save_gif:
        ani.save(filename, writer='pillow', fps=1000//interval)
        print(f"Animation saved as {filename}")
    
    plt.close()  # 정적 그림 표시 방지
    return HTML(ani.to_jshtml())

def save_animation_as_gif(history, Box, filename='simulation.gif', interval=50, frame_skip=1):
    """시뮬레이션 결과를 GIF 파일로 저장합니다."""
    return create_animation(history, interval=interval, save_gif=True, filename=filename, frame_skip=frame_skip)

In [12]:


# 애니메이션 생성 및 표시
animation = create_animation(jax.tree.map(lambda x : x[-1000:], env_state), interval=10, save_gif=True, filename='simulation.gif', frame_skip=10)
animation

Animation saved as simulation.gif
